# Task 2: Improving Fine-Tuning Results

Objective: Improve CIFAR-10 transfer-learning fine-tuning performance in PyTorch using combinations of suggested tutorial methods.

## Baseline Summary
Baseline uses ResNet50 with pretrained ImageNet weights, head replacement for 10 classes, and light fine-tuning on the last block.
This gives a reference point before trying improvement combinations.

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_ds = datasets.CIFAR10('./data', train=False, download=True, transform=transform)

train_len = int(0.9 * len(train_ds))
val_len = len(train_ds) - train_len
train_split, val_split = random_split(
    train_ds, [train_len, val_len], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_split, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_split, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)


In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(train_mode):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train_mode:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train_mode:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_with_best(model, train_loader, val_loader, criterion, optimizer, epochs=5, patience=None, tag='exp'):
    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, device, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, device)
        print(f'[{tag}] Epoch {epoch}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if patience is not None and wait >= patience:
            print(f'[{tag}] Early stopping triggered at epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    return model

def build_resnet50(num_classes=10):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


In [ ]:
# Baseline: freeze all except FC, then unfreeze layer4+fc with standard settings
baseline = build_resnet50().to(device)
for p in baseline.parameters():
    p.requires_grad = False
for p in baseline.fc.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_head = optim.Adam((p for p in baseline.parameters() if p.requires_grad), lr=1e-3)
_ = train_with_best(baseline, train_loader, val_loader, criterion, optimizer_head, epochs=3, tag='baseline-head')

for name, p in baseline.named_parameters():
    p.requires_grad = name.startswith('layer4') or name.startswith('fc')

optimizer_ft = optim.Adam((p for p in baseline.parameters() if p.requires_grad), lr=1e-4)
baseline_acc = train_with_best(baseline, train_loader, val_loader, criterion, optimizer_ft, epochs=5, tag='baseline-ft')
_, baseline_acc = run_epoch(baseline, test_loader, criterion, device)
print('Baseline test accuracy:', baseline_acc)


## Experiment Combination 1
**Methods used:** lower learning rate + early stopping.

In [ ]:
combo1 = build_resnet50().to(device)
for p in combo1.parameters():
    p.requires_grad = False

for name, p in combo1.named_parameters():
    if name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_combo1 = optim.Adam((p for p in combo1.parameters() if p.requires_grad), lr=5e-5)
combo1 = train_with_best(
    combo1, train_loader, val_loader, criterion, optimizer_combo1, epochs=12, patience=3, tag='combo1'
)
_, combo1_acc = run_epoch(combo1, test_loader, criterion, device)
print('Combination 1 test accuracy:', combo1_acc)


**Expected improvement (Combination 1):**
- A lower learning rate usually makes weight updates in pretrained layers more stable, reducing the risk of destroying useful ImageNet features.
- Early stopping prevents unnecessary extra epochs once validation accuracy plateaus, which can reduce overfitting and preserve the best checkpoint.

## Experiment Combination 2
**Methods used:** unfreeze more layers + overfitting prevention (dropout + weight decay).

In [ ]:
combo2 = build_resnet50().to(device)
# Add dropout in the classification head for regularization
combo2.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(combo2.fc.in_features, 10)
)
combo2 = combo2.to(device)

for p in combo2.parameters():
    p.requires_grad = False

# Unfreeze more layers than baseline: layer3, layer4, and fc
for name, p in combo2.named_parameters():
    if name.startswith('layer3') or name.startswith('layer4') or name.startswith('fc'):
        p.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer_combo2 = optim.Adam((p for p in combo2.parameters() if p.requires_grad), lr=1e-4, weight_decay=1e-4)
combo2 = train_with_best(combo2, train_loader, val_loader, criterion, optimizer_combo2, epochs=10, tag='combo2')
_, combo2_acc = run_epoch(combo2, test_loader, criterion, device)
print('Combination 2 test accuracy:', combo2_acc)


**Expected improvement (Combination 2):**
- Unfreezing additional deeper layers (layer3 + layer4) gives the model more flexibility to adapt features to CIFAR-10.
- Dropout and weight decay help regularize the larger trainable parameter set, reducing overfitting while still allowing stronger adaptation.

## Final Comparison / Conclusion
Use the printed `baseline_acc`, `combo1_acc`, and `combo2_acc` to compare outcomes directly.
In practice, Combination 1 often improves training stability, while Combination 2 can achieve higher ceiling performance when regularization is strong enough to control overfitting.